# UE with UQLM



## White-Box Scorers (Single Generation)

**1. Sequence Probability (raw)**
$$\text{SP} = \prod_{t=1}^{T} p(y_t \mid y_{<t})$$
```python
probs = exp(logprobs)        # token probabilities
seq_prob = prod(probs)       # multiply all tokens → full-sequence likelihood
```
Multiply together the probability of every token in the response. *Intuition: literal probability of the whole output — but shrinks with length, so longer responses always score lower.*


---

**2. Sequence Probability (length-normalized)**
$$\text{LNSP} = \exp\!\left(\frac{1}{T}\sum_{t=1}^{T} \log p(y_t \mid y_{<t})\right)$$
```python
norm_prob = exp(mean(logprobs))   # average log-prob per token → length-fair confidence
```
Average the log-probabilities across all T tokens and exponentiate. *Intuition: same as raw sequence probability, but dividing by length T makes short and long responses comparable on equal footing.*


---

**3. Min Token Probability**
$$\text{MinP} = \min_t\, p(y_t \mid y_{<t})$$
```python
min_prob = min(exp(logprobs))     # weakest token probability
```
Take the single lowest token probability in the entire response. *Intuition: one weak link breaks the chain — a single shaky token flags a potential hallucination.*


---

**4. Mean Top-K Token Negentropy**
$$\text{MeanNeg} = \frac{1}{T}\sum_{t=1}^{T} \left(1 - \frac{-\sum_{i \in \mathcal{I}^K_t} p^K_{t,i} \log p^K_{t,i}}{\log K}\right)$$
```python
p   = softmax(top_logprobs)       # normalize top-K alternatives
H   = -sum(p * log(p))            # token entropy
neg = 1 - H / log(K)              # normalize to, higher = confident[1]
mean_token_negentropy = mean(neg) # average confidence across tokens
```
At each token position, take the top-K most probable candidates, rescale them to sum to 1, and compute their entropy. Divide by log(K) to normalize to [0,1], then subtract from 1 so that high values mean confident. Average across all T positions.

**Example** — after *"The capital of France is..."*:
- **Confident:** "Paris" = 0.97, "Lyon" = 0.02, "Nice" = 0.01 → entropy ≈ 0 → negentropy ≈ 1
- **Uncertain:** "Paris" = 0.35, "Lyon" = 0.33, "Nice" = 0.32 → entropy ≈ max → negentropy ≈ 0

*Intuition: how sharp the model's token distributions are on average — 1 means always decisive, 0 means always a coin flip.*

---

**5. Min Top-K Token Negentropy**
$$\text{MinNeg} = \min_t\left(1 - \frac{-\sum_{i \in \mathcal{I}^K_t} p^K_{t,i} \log p^K_{t,i}}{\log K}\right)$$
```python
min_token_negentropy = min(neg)   # weakest token step
```
Same calculation as above at every token position — but instead of averaging, keep only the single worst (least confident) position. *Intuition: finds the most uncertain decision point — if even one token was a near-coin-flip, that moment becomes the score for the whole response.*

---

**6. Probability Margin**
$$\text{PM} = \frac{1}{T}\sum_{t=1}^{T}\left(p^{(1)}_t - p^{(2)}_t\right)$$
```python
p = sort(exp(top_logprobs), desc=True)   # top-K probs per token
margin = p - p                     # gap between best and runner-up[1]
prob_margin = mean(margin)               # average decisiveness
```
At each token position, sort the top-K candidates by probability and subtract the second highest from the highest. Average this gap across all T positions. *Intuition: a large gap means the model decisively chose at each step; a small gap means it was nearly a coin flip throughout.*

---

## White-Box Scorers (Multi-Generation)

**7. Monte Carlo Sequence Probability**
$$\widehat{P}(x) = \frac{1}{M+1}\sum_{m=0}^{M} \text{seq\_prob}(y^{(m)})$$
*(m=0 is the original response, m=1..M are the sampled responses; total M+1 items.)*
```python
seq_probs = [sequence_prob(r) for r in original + samples]   # per-sample confidence
monte_carlo_probability = mean(seq_probs)                     # average over samples
```
Sample M responses stochastically, compute the sequence probability of each, then average. *Intuition: smooths out randomness by averaging confidence across multiple generations — more robust than a single draw.*

---

**8. Consistency and Confidence (CoCoA)**
$$U_{\text{CoCoA}}(y \mid x) = \text{seq\_prob}(y) \cdot \frac{1}{M}\sum_{i=1}^{M} \frac{1 + \cos(y,\, y_i)}{2}$$
*(UQLM normalizes each cosine to [0,1] via `0.5 + cos/2` before averaging; `seq_prob(y)` is length‑normalized when `length_normalize=True`.)*
```python
cosine_scores   = cosine_similarity(response, sampled_responses)   # semantic agreement
response_prob   = sequence_prob(response)                          # confidence of original
consistency_and_confidence = cosine_scores * response_prob
```
Compute the cosine similarity between the original response and each of the sampled responses (via embeddings), then average those similarities. Multiply by the sequence probability of the original response (length‑normalized if enabled). *Intuition: reliable answers are both consistent with their samples and confident in the original output; if either is low, the score is low.*

---

**9. Semantic Negentropy**
$$\text{SemNeg} = 1 - \frac{-\sum_{c \in C} p(c) \log p(c)}{\log(M + 1)}$$
```python
cluster_probs       = P(cluster)                    # NLI-based semantic clusters
H                   = -sum(p * log(p))              # semantic entropy
norm                = H / log(num_responses + 1)    # normalize to[1]
semantic_negentropy = 1 - norm                      # convert to confidence
```
Generate M sampled responses plus the original response, then cluster all responses by meaning using a bidirectional NLI model (mutual entailment). Compute a probability for each cluster by summing the response probabilities of its members (response probability is the sequence probability, length‑normalized if enabled). Compute semantic entropy over these cluster probabilities, normalize by log(M+1), then invert to get a confidence score: *Intuition: if samples agree on the same meaning, one cluster dominates → low entropy → high semantic negentropy (confident). If samples split across meanings, entropy is high → low semantic negentropy (uncertain).*

---

**10. Semantic Density**
$$
\mathrm{SD}(y^*) \;=\; \frac{\sum_{i=1}^{M} w_i \, K(d_i)}{\sum_{i=1}^{M} w_i}
$$

Where:  
$w_i$ = response probability of sample $i$ (from logprobs, length‑normalized if enabled)  
$d_i$ = NLI‑based semantic distance between original response $y^*$ and sample $y_i$  
$K(d_i) = (1 - d_{\text{sq},i}) \cdot \mathbb{1}[d_i \le 1]$, with $d_i = P_c(y^*, y_i) + \tfrac{\sqrt{2}}{2}\,P_n(y^*, y_i)$ and $d_{\text{sq},i} = P_c(y^*, y_i) + \tfrac{1}{2}P_n(y^*, y_i)$ (NLI contradiction/neutral probabilities)

*Note: weights run only over the M sampled responses; the original response $y^*$ is the reference, not a weighted sample.*


```python
# --- Semantic Density (UQLM) ---
weights = tokenprob_response_probabilities      # confidence (sequence probs) for each sampled response
kernel_vals = kernel_values                     # NLI‑based closeness of each sample to the original response
semantic_density = weighted_average(kernel_vals, weights)  # weighted mean closeness
```
UQLM computes semantic density by comparing each sampled response to the original response using an NLI model, turning entailment/neutral/contradiction into a closeness score. These closeness scores are then averaged with weights equal to each sample’s response probability (length‑normalized if enabled). High density means the most likely samples stay semantically close to the original answer; low density means the likely samples drift in meaning, indicating higher uncertainty.

---

## Judge-Based Scorers

**11. P(True)**
$$P(\text{True}) = \begin{cases} \exp(\text{logprob}) & \text{if token starts with "true"} \\ 1 - \exp(\text{logprob}) & \text{if token starts with "false"} \end{cases}$$
```python
if token.startswith("true"):  p_true = exp(logprob)       # prob answer is correct
if token.startswith("false"): p_true = 1 - exp(logprob)   # prob answer is incorrect
```
A separate judge model is asked whether the response is correct. Extract the probability it assigns to the first token being "true" or "false" and convert to a confidence score accordingly. *Intuition: the judge model estimates correctness — if it says "true" with high probability, the response is likely reliable.*


In [17]:
import requests
import pandas as pd
from collections import defaultdict

# Host -> services
LEGAL_SERVICE_URL = "http://localhost:8000/api/v1"
DATA_SERVICE_HOST_URL = "http://localhost:8002"

# Container -> data_service (for legal_service to fetch)
DATA_SERVICE_INTERNAL_URL = "http://data_service:8002"

TEST_MAP = {
    "Lehrlingsbetrieb": "test_lehrlingsausbildung",
    #"Gleichbehandlung ausländischer Anbieter": "test_gleichbehandlung",
    #"Gesamtsumme": "test_gesamtsumme",
    #"Preisspanne": "test_preisspanne",
    #"Keine Preisangabe": "test_keine_preisangabe",
    #"Falsche Rechtsmittelbelehrung": "test_falsche_rechtsmittelbelehrung",
    #"Inkonsistenz Termin": "test_inkonsistenz_termin",
    #"Doppelte Bewertung": "test_doppelte_bewertung",
    #"Verzerrung Bewertung": "test_verzerrung_bewertung",
    #"Keine Gewichtung": "test_keine_gewichtung",
    #"Sachfremde Kriterien": "test_sachfremde_kriterien",
    #"Gerichtsferien": "test_gerichtsferien",
    #"Unberechtigte Referenz": "test_unberechtigte_referenz",
    #"Preisgewichtung": "test_preisgewichtung",
    #"Preis nicht als Kriterium": "test_preis_nicht_als_kriterium",
}

# load GT table once
gt_df = pd.read_csv("sample.csv")

# use all project IDs from the GT table
PROJECT_IDS = gt_df["GT table"].astype(int).tolist()

USE_ALL = True # flip this only

if USE_ALL:
    PROJECT_IDS = gt_df["GT table"].astype(int).tolist()
else:
    PROJECT_IDS = PROJECT_IDS = [191846, 169402, 189879, 235296, 258830, 190944, 243602, 205270, 229691, 221633, 9583, 196565, 192494]





In [19]:
def build_version_map():
    resp = requests.get(f"{DATA_SERVICE_HOST_URL}/projects")
    resp.raise_for_status()
    projects = resp.json()["data"]["projects"]
    versions = defaultdict(list)
    for p in projects:
        versions[p["project_id"]].append(p["simap_version"])
    return versions

version_map = build_version_map()

def get_simap_version(project_id):
    vals = version_map.get(project_id, [])
    if not vals:
        return "simap"
    return "simap_v2" if "simap_v2" in vals else vals[0]

def run_tests_for_project(project_id):
    simap_version = get_simap_version(project_id)
    project_data_url = (
        f"{DATA_SERVICE_INTERNAL_URL}/api/v1/projects/{project_id}"
        f"?simap_version={simap_version}"
    )
    payload = {
        "project_data_url": project_data_url,
        "test_codes": list(TEST_MAP.values()),
    }
    resp = requests.post(f"{LEGAL_SERVICE_URL}/tests/run", json=payload)
    if not resp.ok:
        print("STATUS:", resp.status_code)
        print("BODY:", resp.text[:2000])
    resp.raise_for_status()
    return resp.json()





In [20]:
import math, pandas as pd

def logprobs_table(logprobs, max_tokens=50):
    rows = []
    for i, t in enumerate((logprobs or {}).get("content", [])[:max_tokens]):
        lp = t.get("logprob")
        rows.append({
            "i": i,
            "token": repr(t.get("token")),
            "logprob": lp,
            "prob": math.exp(lp) if isinstance(lp, (int, float)) else None
        })
    return pd.DataFrame(rows)


In [25]:
run_tests_for_project(PROJECT_IDS[0])


{'status': 'success',
 'data': {'results': [{'test_code': 'test_lehrlingsausbildung',
    'simap_project_number': None,
    'violation_detected': True,
    'message': {'title': 'Test test_lehrlingsausbildung - Information',
     'content': 'Keine detaillierten Informationen verfügbar.'},
    'execution_info': {'execution_time_ms': 9370.53108215332,
     'model_used': 'openai/gpt-oss-20b',
     'documents': ['1819_schu_ausschreibung_bkp_211.0_baumeisterarbeiten.pdf',
      '1819_schu_ausschreibung_bkp_211.0_baumeisterarbeiten.pdf (Seite 1)'],
     'prompt_tokens': 1463,
     'completion_tokens': 670,
     'cost_usd': None,
     'checks_executed': [{'check_code': 'staatsvertragsbereich',
       'version': '1.0.0',
       'result': True,
       'llm_answer': 'Zuerst habe ich den Text im Feld „procedure_details“ gelesen.  \nDer Text enthält die Zeile „Staatsvertragsbereich: Ja“.  \nDa die Angabe exakt mit „Ja“ übereinstimmt, liegt das Projekt im Staatsvertragsbereich.  \nDaher ist die Bedi

In [ ]:
# =========================
# White-Box UQ Pipeline (strict no-NaN + cache NaN recalc)
# =========================
# Requires pre-existing notebook objects/functions:
# - PROJECT_IDS, TEST_MAP, gt_df, run_tests_for_project, get_simap_version

import os
import re
import sys
import json
import time
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Any, Dict, List, Optional
from dotenv import load_dotenv
from requests import HTTPError

# Optional Google auth imports (only needed for gemini_vertex backend)
try:
    import google.auth
    from google.auth.transport.requests import Request as GoogleAuthRequest
except Exception:
    google = None
    GoogleAuthRequest = None


# -------------------------
# 0) Path + environment
# -------------------------
def _first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None


def _sanitize_tag(s: str) -> str:
    return "".join(ch if ch.isalnum() or ch in ("_", "-", ".") else "_" for ch in s)


def _resolve_legal_service_root() -> Path:
    candidates = [
        Path("../intelliprocure-ai-legal/legal_service"),
        Path("intelliprocure-ai-legal/legal_service"),
        Path("../../intelliprocure-ai-legal/legal_service"),
    ]
    root = _first_existing(candidates)
    if root is None:
        raise FileNotFoundError("Could not locate intelliprocure-ai-legal/legal_service.")
    return root


def _resolve_ckpt_dir() -> Path:
    candidates = [
        Path("_ckpt"),
        Path("uncertainty_estimation/_ckpt"),
        Path("uncertainty_estimation/uncertainty_estimation/_ckpt"),
    ]
    existing = _first_existing(candidates)
    if existing is not None:
        existing.mkdir(parents=True, exist_ok=True)
        return existing
    default = Path("uncertainty_estimation/_ckpt")
    default.mkdir(parents=True, exist_ok=True)
    return default


LEGAL_SERVICE_ROOT = _resolve_legal_service_root()
env_path = LEGAL_SERVICE_ROOT / ".env"
if env_path.exists():
    load_dotenv(env_path, override=True)
else:
    print("Warning: .env not found under legal_service; using current process env vars.")


# -------------------------
# 1) Backend config
# -------------------------
LLM_BACKEND = os.environ["LLM_BACKEND"].strip()

if LLM_BACKEND == "hf":
    LLM_BASE_URL = "https://router.huggingface.co/v1"
    LLM_API_KEY = os.environ["HF_TOKEN"]
    LLM_MODEL = os.environ["HF_MODEL"]

elif LLM_BACKEND == "openrouter":
    LLM_BASE_URL = "https://openrouter.ai/api/v1"
    LLM_API_KEY = os.environ["OPENROUTER_API_KEY"]
    LLM_MODEL = os.environ["OPENROUTER_MODEL"]

elif LLM_BACKEND == "together":
    LLM_BASE_URL = "https://api.together.xyz/v1"
    LLM_API_KEY = os.environ["TOGETHER_API_KEY"]
    LLM_MODEL = os.getenv("TOGETHER_MODEL")
    TOGETHER_LOGPROBS_K = int(os.getenv("TOGETHER_LOGPROBS_K", "5"))

elif LLM_BACKEND == "gemini_vertex":
    VERTEX_PROJECT_ID = os.environ["VERTEX_PROJECT_ID"]
    VERTEX_LOCATION = os.environ["VERTEX_LOCATION"]
    LLM_MODEL = os.environ["GEMINI_MODEL"]
    LLM_BASE_URL = f"https://{VERTEX_LOCATION}-aiplatform.googleapis.com/v1"
    LLM_API_KEY = None

else:
    raise ValueError(
        f"LLM_BACKEND must be 'hf', 'openrouter', 'together', or 'gemini_vertex', got: {LLM_BACKEND}"
    )

DATA_SERVICE_API_KEY = os.getenv("DATA_SERVICE_API_KEY", "")
DATA_SERVICE_HOST_URL = "http://localhost:8002"

print("LLM_BACKEND:", LLM_BACKEND)
print("LLM_MODEL:", LLM_MODEL)


# -------------------------
# 2) Run config
# -------------------------
ENABLE_SAMPLED = True
ENABLE_PTRUE = True

NUM_SAMPLES = 5
SAMPLING_TEMPERATURE = 0.7
TOP_LOGPROBS = 5
MAX_TOKENS = 4000
NLI_MAX_LENGTH = 4000

MAX_SAMPLE_TRIES_FACTOR = 3
MAX_PROJECT_RETRIES = 3

MAX_UPSTREAM_RETRIES = 10
BACKOFF_BASE = 2.0
BACKOFF_CAP = 180.0

MAX_CHAT_RETRIES = 8
CHAT_BACKOFF_BASE = 1.5
CHAT_BACKOFF_CAP = 120.0

PROJECT_COOLDOWN_SEC = 2.0

# p_true settings
PTRUE_MAX_TOKENS = 512
PTRUE_TEMPERATURE = 0.0

RUN_TAG = "VerzerrungBewertung_GPT20B"
ACTIVE_TEST_CODES = {"test_verzerrung_bewertung"}  # set None to run all tests

CONTROL_TOK_RE = re.compile(r"^<\|.*\|>$")
LOGPROB_FLOOR = -80.0

EXPECTED_SCORE_COLS = [
    "sequence_probability",
    "min_probability",
    "mean_token_negentropy",
    "min_token_negentropy",
    "probability_margin",
    "semantic_negentropy",
    "semantic_density",
    "monte_carlo_probability",
    "consistency_and_confidence",
    "p_true",
]
NON_PTRUE_SCORE_COLS = [c for c in EXPECTED_SCORE_COLS if c != "p_true"]
TOP_COLS = ["mean_token_negentropy", "min_token_negentropy", "probability_margin"]
SAMPLED_COLS = ["semantic_negentropy", "semantic_density", "monte_carlo_probability", "consistency_and_confidence"]
STRICT_REQUIRE_ALL_METRICS = True
RECALC_LEGACY_ZERO_TOP = True

BASE_OUTPUT_COLS = ["project_id", "test_code", "check_code", "gt", "pred", "match", "response"]
REQUIRED_OUTPUT_COLS = BASE_OUTPUT_COLS + EXPECTED_SCORE_COLS


# -------------------------
# 3) Load UQLM
# -------------------------
sys.path.append("uqlm-main")

from uqlm.white_box.single_logprobs import SingleLogprobsScorer, SINGLE_LOGPROBS_SCORER_NAMES
from uqlm.white_box.top_logprobs import TopLogprobsScorer, TOP_LOGPROBS_SCORER_NAMES
from uqlm.white_box.sampled_logprobs import SampledLogprobsScorer, SAMPLED_LOGPROBS_SCORER_NAMES


# -------------------------
# 4) Checkpoint paths
# -------------------------
CKPT_DIR = _resolve_ckpt_dir()
_TAG = _sanitize_tag(RUN_TAG)
WB_ROWS_PATH = CKPT_DIR / f"final_df_wb_partial_{_TAG}.parquet"
WB_FAILS_PATH = CKPT_DIR / f"failed_project_ids_wb_{_TAG}.json"


# -------------------------
# 5) Helpers
# -------------------------
def _ensure_score_schema(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in EXPECTED_SCORE_COLS:
        if c not in out.columns:
            out[c] = np.nan
    return out


def _ensure_output_schema(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in BASE_OUTPUT_COLS:
        if c not in out.columns:
            out[c] = np.nan
    out = _ensure_score_schema(out)
    out = out[REQUIRED_OUTPUT_COLS].copy()
    return out


def _sanitize_scores(df: pd.DataFrame, fill_ptrue_if_missing: bool = False) -> pd.DataFrame:
    out = _ensure_score_schema(df)
    out[EXPECTED_SCORE_COLS] = (
        out[EXPECTED_SCORE_COLS]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )
    if fill_ptrue_if_missing:
        out["p_true"] = out["p_true"].fillna(0.0)
    return out


def _safe_logprob(v):
    try:
        x = float(v)
    except (TypeError, ValueError):
        return None
    if not np.isfinite(x):
        return None
    return float(np.clip(x, LOGPROB_FLOOR, 0.0))


def _strip_think_blocks(text: str) -> str:
    if not isinstance(text, str) or not text:
        return ""
    return re.sub(r"<think>.*?</think>", " ", text, flags=re.IGNORECASE | re.DOTALL).strip()


def extract_message_text(choice):
    msg = (choice.get("message") or {})
    content = msg.get("content")

    if isinstance(content, str) and content.strip():
        return content.strip()

    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict):
                t = block.get("text")
                if isinstance(t, str) and t.strip():
                    parts.append(t.strip())
        if parts:
            return "\n".join(parts).strip()

    rc = msg.get("reasoning_content")
    if isinstance(rc, str) and rc.strip():
        return rc.strip()

    r = msg.get("reasoning")
    if isinstance(r, str) and r.strip():
        return r.strip()

    return ""


def _is_transient_http_error(e: HTTPError) -> bool:
    resp = getattr(e, "response", None)
    status = getattr(resp, "status_code", None)
    body = ""
    try:
        body = (resp.text or "").lower() if resp is not None else ""
    except Exception:
        pass

    if status in {429, 502, 503, 504}:
        return True

    if status == 500:
        markers = [
            "resource exhausted",
            "rate limit",
            "quota",
            "too many requests",
            "temporarily unavailable",
        ]
        return any(m in body for m in markers)

    return False


def _choice_has_logprobs(choice: Dict[str, Any]) -> bool:
    lp = choice.get("logprobs")
    if isinstance(lp, dict):
        content = lp.get("content")
        return isinstance(content, list) and len(content) > 0
    if isinstance(lp, list):
        return len(lp) > 0
    return False


def _require_logprobs_in_response(data: Dict[str, Any], context: str = ""):
    choice = (data.get("choices") or [{}])[0]
    if not _choice_has_logprobs(choice):
        raise ValueError(f"Missing logprobs in successful response ({context}).")


_VERTEX_CREDS = None


def _vertex_access_token(force_refresh: bool = False):
    if google is None or GoogleAuthRequest is None:
        raise RuntimeError("google-auth is required for gemini_vertex backend.")
    global _VERTEX_CREDS
    if _VERTEX_CREDS is None:
        _VERTEX_CREDS, _ = google.auth.default(
            scopes=["https://www.googleapis.com/auth/cloud-platform"]
        )
    if force_refresh or (not _VERTEX_CREDS.valid) or (_VERTEX_CREDS.token is None):
        _VERTEX_CREDS.refresh(GoogleAuthRequest())
    return _VERTEX_CREDS.token


def _gemini_extract_text(candidate):
    content = candidate.get("content") or {}
    parts = content.get("parts") or []
    texts = []
    for p in parts:
        t = p.get("text")
        if isinstance(t, str) and t.strip():
            texts.append(t)
    return "".join(texts).strip()


def _gemini_logprobs_to_openai_content(logprobs_result):
    if not isinstance(logprobs_result, dict):
        return []

    top_steps = logprobs_result.get("topCandidates") or []
    chosen_steps = logprobs_result.get("chosenCandidates") or []
    out = []

    n = max(len(top_steps), len(chosen_steps))
    for i in range(n):
        chosen = chosen_steps[i] if i < len(chosen_steps) else {}
        top_obj = top_steps[i] if i < len(top_steps) else {}
        top_list = top_obj.get("candidates") or []

        token = chosen.get("token")
        logp = _safe_logprob(chosen.get("logProbability"))
        tokid = chosen.get("tokenId")

        if (token is None or logp is None) and top_list:
            best = top_list[0]
            token = best.get("token")
            logp = _safe_logprob(best.get("logProbability"))
            tokid = best.get("tokenId")

        if not isinstance(token, str) or logp is None:
            continue

        top_clean = []
        for alt in top_list:
            alt_tok = alt.get("token")
            alt_lp = _safe_logprob(alt.get("logProbability"))
            if not isinstance(alt_tok, str) or alt_lp is None:
                continue
            top_clean.append(
                {"token": alt_tok, "logprob": alt_lp, "token_id": alt.get("tokenId")}
            )

        out.append(
            {"token": token, "logprob": logp, "token_id": tokid, "top_logprobs": top_clean}
        )

    return out


def _messages_to_gemini_payload(messages, temperature, max_tokens, enable_logprobs, top_logprobs):
    contents = []
    system_texts = []

    for m in messages:
        role = (m.get("role") or "").lower()
        content = m.get("content", "")

        if isinstance(content, list):
            text_parts = []
            for c in content:
                if isinstance(c, dict):
                    t = c.get("text")
                    if isinstance(t, str) and t.strip():
                        text_parts.append(t)
            text = "\n".join(text_parts).strip()
        else:
            text = str(content).strip()

        if not text:
            continue

        if role == "system":
            system_texts.append(text)
        else:
            gem_role = "model" if role == "assistant" else "user"
            contents.append({"role": gem_role, "parts": [{"text": text}]})

    generation_config = {
        "temperature": float(temperature),
        "maxOutputTokens": int(max_tokens),
    }
    if enable_logprobs:
        generation_config["responseLogprobs"] = True
        generation_config["logprobs"] = int(top_logprobs)

    payload = {
        "contents": contents if contents else [{"role": "user", "parts": [{"text": ""}]}],
        "generationConfig": generation_config,
    }

    if system_texts:
        payload["systemInstruction"] = {
            "role": "system",
            "parts": [{"text": "\n\n".join(system_texts)}],
        }

    return payload


def _vertex_chat(messages, temperature=0.1, max_tokens=MAX_TOKENS, logprobs=True, top_logprobs=TOP_LOGPROBS):
    url = (
        f"{LLM_BASE_URL}/projects/{VERTEX_PROJECT_ID}/locations/{VERTEX_LOCATION}"
        f"/publishers/google/models/{LLM_MODEL}:generateContent"
    )

    payload = _messages_to_gemini_payload(
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
        enable_logprobs=logprobs,
        top_logprobs=top_logprobs,
    )

    token = _vertex_access_token(force_refresh=False)
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    r = requests.post(url, json=payload, headers=headers, timeout=120)

    if r.status_code in (401, 403):
        token = _vertex_access_token(force_refresh=True)
        headers["Authorization"] = f"Bearer {token}"
        r = requests.post(url, json=payload, headers=headers, timeout=120)

    if r.status_code >= 400:
        raise HTTPError(f"{r.status_code} {r.reason}: {r.text[:1200]}", response=r)

    j = r.json()
    cand = (j.get("candidates") or [{}])[0]
    text = _gemini_extract_text(cand)
    content_logprobs = _gemini_logprobs_to_openai_content(cand.get("logprobsResult"))

    usage = j.get("usageMetadata") or {}
    out = {
        "choices": [
            {
                "message": {"role": "assistant", "content": text},
                "logprobs": {"content": content_logprobs},
                "finish_reason": (cand.get("finishReason") or "").lower() or None,
            }
        ],
        "usage": {
            "prompt_tokens": usage.get("promptTokenCount"),
            "completion_tokens": usage.get("candidatesTokenCount", usage.get("thoughtsTokenCount")),
            "total_tokens": usage.get("totalTokenCount"),
        },
        "model": j.get("modelVersion", LLM_MODEL),
    }

    if logprobs:
        _require_logprobs_in_response(out, context="vertex")
    return out


def llm_chat(messages, temperature=0.1, max_tokens=MAX_TOKENS, logprobs=True, top_logprobs=TOP_LOGPROBS):
    if LLM_BACKEND == "gemini_vertex":
        return _vertex_chat(
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
            logprobs=logprobs,
            top_logprobs=top_logprobs,
        )

    payload = {
        "model": LLM_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    headers = {
        "Authorization": f"Bearer {LLM_API_KEY}",
        "HTTP-Referer": "https://intelliprocure.ch",
        "X-Title": "UQLM White-Box Eval",
    }

    if LLM_BACKEND == "together":
        if logprobs:
            payload["logprobs"] = int(TOGETHER_LOGPROBS_K)
        headers = {
            "Authorization": f"Bearer {LLM_API_KEY}",
            "Content-Type": "application/json",
        }
        r = requests.post(f"{LLM_BASE_URL}/chat/completions", json=payload, headers=headers, timeout=120)
        if r.status_code >= 400:
            raise HTTPError(f"{r.status_code} {r.reason}: {r.text[:1200]}", response=r)
        data = r.json()
        if logprobs:
            _require_logprobs_in_response(data, context="together")
        return data

    if logprobs:
        payload["logprobs"] = True
        payload["top_logprobs"] = int(top_logprobs)

    if LLM_BACKEND == "openrouter":
        payload["provider"] = {"require_parameters": True}

    r = requests.post(f"{LLM_BASE_URL}/chat/completions", json=payload, headers=headers, timeout=120)
    if r.status_code >= 400:
        raise HTTPError(f"{r.status_code} {r.reason}: {r.text[:1200]}", response=r)

    data = r.json()
    if logprobs:
        _require_logprobs_in_response(data, context=LLM_BACKEND)
    return data


def llm_chat_with_backoff(messages, temperature=0.1, max_tokens=MAX_TOKENS, logprobs=True, top_logprobs=TOP_LOGPROBS):
    last_err = None
    for attempt in range(MAX_CHAT_RETRIES):
        try:
            return llm_chat(
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
                logprobs=logprobs,
                top_logprobs=top_logprobs,
            )
        except (HTTPError, ValueError) as e:
            last_err = e
            if isinstance(e, HTTPError) and not _is_transient_http_error(e):
                raise
            wait = min(CHAT_BACKOFF_CAP, CHAT_BACKOFF_BASE * (2 ** attempt)) + np.random.uniform(0, 0.6)
            print(f"llm_chat transient error; retry {attempt+1}/{MAX_CHAT_RETRIES} in {wait:.1f}s")
            time.sleep(wait)
    raise last_err


def llm_chat_require_logprobs(messages, temperature=0.1, max_tokens=50):
    last_err = None
    for t in (temperature, 0.2):
        try:
            return llm_chat_with_backoff(
                messages=messages,
                temperature=t,
                max_tokens=max_tokens,
                logprobs=True,
                top_logprobs=TOP_LOGPROBS,
            )
        except (HTTPError, ValueError) as e:
            last_err = e
    raise last_err


def to_uqlm_logprobs(logprobs_payload):
    if logprobs_payload is None:
        return []

    if isinstance(logprobs_payload, dict):
        toks = logprobs_payload.get("content") or []
    elif isinstance(logprobs_payload, list):
        toks = logprobs_payload
    else:
        return []

    cleaned = []
    final_only = []
    expect_channel_name = False
    in_final = False

    for t in toks:
        tok = (t.get("token") or "").strip()
        if not tok:
            continue

        if tok == "<|channel|>":
            expect_channel_name = True
            in_final = False
            continue

        if expect_channel_name:
            in_final = (tok.lower() == "final")
            expect_channel_name = False
            continue

        if CONTROL_TOK_RE.match(tok):
            continue

        lp = _safe_logprob(t.get("logprob"))
        if lp is None:
            continue

        top_clean = []
        for alt in (t.get("top_logprobs") or []):
            alt_tok = (alt.get("token") or "").strip()
            alt_lp = _safe_logprob(alt.get("logprob"))
            if not alt_tok or alt_lp is None or CONTROL_TOK_RE.match(alt_tok):
                continue
            top_clean.append({**alt, "logprob": alt_lp})

        row = {**t, "logprob": lp, "top_logprobs": top_clean}
        cleaned.append(row)
        if in_final:
            final_only.append(row)

    return final_only if final_only else cleaned


def _top_tokens_with_alternatives(one_logprobs):
    return [
        t for t in one_logprobs
        if isinstance(t.get("top_logprobs"), list) and len(t["top_logprobs"]) >= 2
    ]


def has_top_logprobs(one_logprobs, min_tokens=5):
    if not one_logprobs:
        return False
    return len(_top_tokens_with_alternatives(one_logprobs)) >= min_tokens


def build_messages(check_code, project_summary):
    checks_cfg = json.loads(
        (LEGAL_SERVICE_ROOT / "test_definitions/checks.json").read_text(encoding="utf-8")
    )
    prompts_dir = LEGAL_SERVICE_ROOT / "test_definitions/prompts_checks"
    system_prompt = (LEGAL_SERVICE_ROOT / "test_definitions/system_prompt.txt").read_text(encoding="utf-8").strip()

    cfg = checks_cfg[check_code]
    required_fields = cfg["required_fields"]
    prompt_file = cfg["prompt_file"]

    def format_project_data(project_data):
        formatted = ""
        for k, v in project_data.items():
            if v is None or v == "":
                v = "No answer"
            elif not isinstance(v, str):
                v = str(v)
            formatted += f"**{k}:**\n{v}\n\n"
        return formatted

    project_data = {f: project_summary.get(f, "") for f in required_fields}
    prompt = (prompts_dir / prompt_file).read_text(encoding="utf-8").replace(
        "{PROJECT_DATA}",
        format_project_data(project_data),
    )

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]


def fetch_project_summary(project_id, simap_version):
    url = f"{DATA_SERVICE_HOST_URL}/api/v1/projects/{project_id}?simap_version={simap_version}"
    headers = {"X-API-KEY": DATA_SERVICE_API_KEY} if DATA_SERVICE_API_KEY else {}
    r = requests.get(url, headers=headers, timeout=60)
    r.raise_for_status()
    return r.json()["data"]["attributes"]["summary"]


def extract_p_true_from_response(data):
    choice = (data.get("choices") or [{}])[0]
    text = extract_message_text(choice)

    m = re.findall(r"result\s*:\s*(true|false|wahr|falsch)\b", text, flags=re.IGNORECASE)
    if m:
        v = m[-1].lower()
        return 1.0 if v in ("true", "wahr") else 0.0

    lp = to_uqlm_logprobs(choice.get("logprobs"))
    in_think = False
    for item in lp:
        tok_raw = (item.get("token") or "")
        tok = tok_raw.strip().lower()
        logp = item.get("logprob")

        if not tok:
            continue
        if tok.startswith("<think"):
            in_think = True
            continue
        if tok.startswith("</think"):
            in_think = False
            continue
        if in_think or logp is None:
            continue

        tok_norm = re.sub(r"[^a-z]", "", tok)
        prob = float(np.exp(float(logp)))

        if tok_norm in ("true", "wahr"):
            return prob
        if tok_norm in ("false", "falsch"):
            return float(1.0 - prob)

    text_no_think = _strip_think_blocks(text)
    m2 = re.findall(r"\b(true|false|wahr|falsch)\b", text_no_think, flags=re.IGNORECASE)
    if m2:
        v = m2[-1].lower()
        return 1.0 if v in ("true", "wahr") else 0.0

    raise ValueError(f"Could not parse strict p_true from verifier output: {text[:300]!r}")


def run_tests_for_project_with_backoff(project_id):
    last_err = None
    for attempt in range(MAX_UPSTREAM_RETRIES):
        try:
            return run_tests_for_project(project_id)
        except HTTPError as e:
            last_err = e
            if not _is_transient_http_error(e):
                raise
            sleep_s = min(BACKOFF_CAP, BACKOFF_BASE * (2 ** attempt)) + np.random.uniform(0, 0.7)
            print(f"run_tests_for_project transient error; retry {attempt+1}/{MAX_UPSTREAM_RETRIES} in {sleep_s:.1f}s")
            time.sleep(sleep_s)
    raise last_err


def _assert_no_nan_output(df: pd.DataFrame, context: str = ""):
    bad = df[REQUIRED_OUTPUT_COLS].isna().any(axis=1)
    if bad.any():
        n_bad = int(bad.sum())
        raise ValueError(f"{context} contains NaN in required output columns for {n_bad} row(s).")


# -------------------------
# 6) Per-project execution
# -------------------------
def run_one_project(project_id):
    resp = run_tests_for_project_with_backoff(project_id)

    gt_row = gt_df.loc[gt_df["GT table"] == project_id]
    if gt_row.empty:
        print("No GT row, skipping", project_id)
        return None
    gt_row = gt_row.iloc[0]

    by_code = {r["test_code"]: r for r in resp["data"]["results"]}
    gt_by_test = {}
    for display_name, test_code in TEST_MAP.items():
        gt = bool(gt_row[display_name])
        pred = bool(by_code.get(test_code, {}).get("violation_detected"))
        gt_by_test[test_code] = (gt == pred)

    records = []
    for r in resp["data"]["results"]:
        if ACTIVE_TEST_CODES and r["test_code"] not in ACTIVE_TEST_CODES:
            continue

        label = gt_by_test.get(r["test_code"])
        for c in r.get("execution_info", {}).get("checks_executed", []):
            records.append(
                {
                    "project_id": project_id,
                    "test_code": r["test_code"],
                    "check_code": c["check_code"],
                    "response": str(c.get("llm_answer", "") or ""),
                    "logprobs": to_uqlm_logprobs(c.get("logprobs")),
                    "label": int(label) if label is not None else None,
                }
            )

    records = [r for r in records if r["logprobs"]]
    if not records:
        raise RuntimeError(f"No valid logprobs records for project_id={project_id}")

    logprobs_results = [rec["logprobs"] for rec in records]

    requested_scorers = [
        "sequence_probability",
        "min_probability",
        "mean_token_negentropy",
        "min_token_negentropy",
        "probability_margin",
    ]
    single_ok = [s for s in requested_scorers if s in SINGLE_LOGPROBS_SCORER_NAMES]
    top_ok = [s for s in requested_scorers if s in TOP_LOGPROBS_SCORER_NAMES]

    # fixed schema from start (strict: keep missing as NaN until truly computed)
    scores_df = pd.DataFrame(index=np.arange(len(records)))
    for c in EXPECTED_SCORE_COLS:
        scores_df[c] = np.nan

    # single-logprob scorers
    single = SingleLogprobsScorer(scorers=single_ok, length_normalize=True)
    single_scores = pd.DataFrame(single.evaluate(logprobs_results))
    single_scores = (
        single_scores.apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )
    for c in single_scores.columns:
        scores_df[c] = single_scores[c].values

    # top-logprob scorers (only where enough tokens)
    top_ready = [_top_tokens_with_alternatives(lp) for lp in logprobs_results]
    top_mask = [has_top_logprobs(lp, min_tokens=5) for lp in top_ready]
    top_available = np.array(top_mask, dtype=bool)
    if top_ok and any(top_mask):
        top_input = [lp for lp, ok in zip(top_ready, top_mask) if ok]
        top = TopLogprobsScorer(scorers=top_ok)
        top_scores = pd.DataFrame(top.evaluate(top_input))
        top_scores = (
            top_scores.apply(pd.to_numeric, errors="coerce")
            .replace([np.inf, -np.inf], np.nan)
        )
        scores_df.loc[top_mask, top_scores.columns] = top_scores.values

    simap_version = get_simap_version(project_id)
    project_summary = fetch_project_summary(project_id, simap_version)

    # sampled scorers
    if ENABLE_SAMPLED:
        prompts = []
        sampled_responses = []
        sampled_logprobs_results = []
        sampled_idx = []
        sampled_available = np.zeros(len(records), dtype=bool)

        for i, rec in enumerate(records):
            msgs = build_messages(rec["check_code"], project_summary)
            prompt_text = msgs[-1]["content"]

            sample_texts = []
            sample_logprobs = []
            tries = 0
            max_tries = NUM_SAMPLES * MAX_SAMPLE_TRIES_FACTOR

            while len(sample_texts) < NUM_SAMPLES and tries < max_tries:
                tries += 1
                try:
                    data = llm_chat_with_backoff(
                        msgs,
                        temperature=SAMPLING_TEMPERATURE,
                        max_tokens=MAX_TOKENS,
                        logprobs=True,
                        top_logprobs=TOP_LOGPROBS,
                    )
                except (HTTPError, ValueError):
                    continue

                choice = (data.get("choices") or [{}])[0]
                text = extract_message_text(choice)
                lp = to_uqlm_logprobs(choice.get("logprobs"))
                if not text or not lp:
                    continue

                sample_texts.append(text)
                sample_logprobs.append(lp)

            if len(sample_texts) == NUM_SAMPLES:
                sampled_idx.append(i)
                sampled_available[i] = True
                prompts.append(prompt_text)
                sampled_responses.append(sample_texts)
                sampled_logprobs_results.append(sample_logprobs)

        if sampled_idx:
            sampled = SampledLogprobsScorer(
                scorers=SAMPLED_LOGPROBS_SCORER_NAMES,
                length_normalize=True,
                prompts_in_nli=True,
                device="cpu",
                max_length=NLI_MAX_LENGTH,
            )
            sampled_scores = pd.DataFrame(
                sampled.evaluate(
                    responses=[records[i]["response"] for i in sampled_idx],
                    sampled_responses=sampled_responses,
                    logprobs_results=[records[i]["logprobs"] for i in sampled_idx],
                    sampled_logprobs_results=sampled_logprobs_results,
                    prompts=prompts,
                )
            )
            sampled_scores = (
                sampled_scores.apply(pd.to_numeric, errors="coerce")
                .replace([np.inf, -np.inf], np.nan)
            )
            scores_df.loc[sampled_idx, sampled_scores.columns] = sampled_scores.values

    # p_true
    else:
        sampled_available = np.zeros(len(records), dtype=bool)

    if ENABLE_PTRUE:
        p_true = np.empty(len(records), dtype=float)

        ptrue_system_prompt = (
            "Du bist ein strikter Verifier.\n"
            "Gib als LETZTE Zeile exakt: RESULT: True oder RESULT: False.\n"
            "Keine weiteren Inhalte nach der RESULT-Zeile."
        )

        for i in range(len(records)):
            msgs = build_messages(records[i]["check_code"], project_summary)
            prompt_text = msgs[-1]["content"]

            ptrue_prompt = (
                f"Frage: {prompt_text}\n"
                f"Antwort: {records[i]['response']}\n\n"
                "Ist die Antwort korrekt?\n"
                "Gib als LETZTE Zeile exakt: RESULT: True oder RESULT: False."
            )

            data = llm_chat_require_logprobs(
                [
                    {"role": "system", "content": ptrue_system_prompt},
                    {"role": "user", "content": ptrue_prompt},
                ],
                temperature=PTRUE_TEMPERATURE,
                max_tokens=PTRUE_MAX_TOKENS,
            )
            p_true[i] = extract_p_true_from_response(data)

        scores_df["p_true"] = p_true
    else:
        scores_df["p_true"] = 0.0

    pred_by_test = {r["test_code"]: r.get("violation_detected") for r in resp["data"]["results"]}
    gt_val_by_test = {test_code: bool(gt_row[display_name]) for display_name, test_code in TEST_MAP.items()}

    records_df = pd.DataFrame(records)
    records_df["gt"] = records_df["test_code"].map(gt_val_by_test)
    records_df["pred"] = records_df["test_code"].map(pred_by_test)
    records_df["match"] = records_df["gt"] == records_df["pred"]

    # strict sanitize + required metric availability
    scores_df = _sanitize_scores(scores_df, fill_ptrue_if_missing=not ENABLE_PTRUE)

    if STRICT_REQUIRE_ALL_METRICS:
        missing_top_idx = np.where(~top_available)[0].tolist()
        missing_sampled_idx = np.where(~sampled_available)[0].tolist() if ENABLE_SAMPLED else []
        missing_ptrue_idx = np.where(scores_df["p_true"].isna().values)[0].tolist() if ENABLE_PTRUE else []
        bad = sorted(set(missing_top_idx + missing_sampled_idx + missing_ptrue_idx))
        if bad:
            preview = ", ".join(f"{records[i]['check_code']}@{records[i]['project_id']}" for i in bad[:8])
            raise RuntimeError(
                f"Strict mode: missing metrics on {len(bad)} row(s). Examples: {preview}"
            )

    if scores_df[EXPECTED_SCORE_COLS].isna().any().any():
        nan_counts = scores_df[EXPECTED_SCORE_COLS].isna().sum()
        raise RuntimeError(
            f"No-NaN rule violated in run_one_project({project_id}): "
            f"{nan_counts[nan_counts > 0].to_dict()}"
        )

    merged = pd.concat([records_df.reset_index(drop=True), scores_df.reset_index(drop=True)], axis=1)
    merged = _ensure_output_schema(merged)
    _assert_no_nan_output(merged, context=f"run_one_project({project_id})")

    return merged


# -------------------------
# 7) Run all projects (checkpointed, per RUN_TAG)
# -------------------------
if WB_ROWS_PATH.exists():
    try:
        final_df_wb = pd.read_parquet(WB_ROWS_PATH)
    except Exception as e:
        print("Could not read WB checkpoint, starting fresh:", e)
        final_df_wb = pd.DataFrame()
else:
    final_df_wb = pd.DataFrame()

if ACTIVE_TEST_CODES and not final_df_wb.empty and "test_code" in final_df_wb.columns:
    final_df_wb = final_df_wb[final_df_wb["test_code"].isin(ACTIVE_TEST_CODES)].copy()

recalc_ids = []
if not final_df_wb.empty:
    final_df_wb = _ensure_output_schema(final_df_wb)
    final_df_wb = _sanitize_scores(final_df_wb, fill_ptrue_if_missing=not ENABLE_PTRUE)

    nan_mask = final_df_wb[REQUIRED_OUTPUT_COLS].isna().any(axis=1)

    legacy_zero_top_mask = pd.Series(False, index=final_df_wb.index)
    if RECALC_LEGACY_ZERO_TOP:
        legacy_zero_top_mask = (final_df_wb[TOP_COLS].fillna(0.0) == 0.0).all(axis=1)

    recalc_mask = nan_mask | legacy_zero_top_mask
    recalc_ids = sorted(
        final_df_wb.loc[recalc_mask, "project_id"].dropna().astype(int).unique().tolist()
    )

    if recalc_ids:
        print(f"Found cached invalid rows. Will recalculate project_ids: {recalc_ids}")
        final_df_wb = final_df_wb.loc[~final_df_wb["project_id"].isin(recalc_ids)].copy()
        final_df_wb.to_parquet(WB_ROWS_PATH, index=False)

done_ids = set(final_df_wb["project_id"].unique()) if (not final_df_wb.empty and "project_id" in final_df_wb.columns) else set()
pending_ids = [pid for pid in PROJECT_IDS if pid not in done_ids]

if WB_FAILS_PATH.exists():
    try:
        failed_ids = json.loads(WB_FAILS_PATH.read_text(encoding="utf-8"))
    except Exception:
        failed_ids = []
else:
    failed_ids = []

failed_ids = [pid for pid in failed_ids if pid in PROJECT_IDS]

print(f"RUN_TAG: {_TAG}")
print(f"ACTIVE_TEST_CODES: {ACTIVE_TEST_CODES}")
print(f"Recovered rows: {len(final_df_wb)}")
print(f"Recalc IDs from cache NaNs: {len(recalc_ids)}")
print(f"Done IDs: {len(done_ids)} | Pending IDs: {len(pending_ids)}")

for pid in pending_ids:
    print("\nRunning", pid)
    ok = False

    for attempt in range(1, MAX_PROJECT_RETRIES + 1):
        try:
            out = run_one_project(pid)

            if out is not None and not out.empty:
                out = out.copy()
                if ACTIVE_TEST_CODES:
                    out = out[out["test_code"].isin(ACTIVE_TEST_CODES)].copy()

                if not out.empty:
                    if not final_df_wb.empty:
                        if ACTIVE_TEST_CODES:
                            mask = (final_df_wb["project_id"] == pid) & (final_df_wb["test_code"].isin(ACTIVE_TEST_CODES))
                        else:
                            mask = (final_df_wb["project_id"] == pid)
                        final_df_wb = final_df_wb.loc[~mask].copy()

                    final_df_wb = pd.concat([final_df_wb, out], ignore_index=True)
                    final_df_wb = final_df_wb.drop_duplicates(
                        subset=["project_id", "test_code", "check_code"], keep="last"
                    )

                    final_df_wb = _ensure_output_schema(final_df_wb)
                    final_df_wb = _sanitize_scores(final_df_wb, fill_ptrue_if_missing=not ENABLE_PTRUE)
                    _assert_no_nan_output(final_df_wb, context=f"checkpoint after pid={pid}")

                    final_df_wb.to_parquet(WB_ROWS_PATH, index=False)
                    print("  saved rows:", len(final_df_wb))
                else:
                    raise RuntimeError(f"Output empty after active test filter for pid={pid}")
            else:
                raise RuntimeError(f"No rows returned for pid={pid}")

            if pid in failed_ids:
                failed_ids = [x for x in failed_ids if x != pid]
                WB_FAILS_PATH.write_text(json.dumps(failed_ids, indent=2), encoding="utf-8")

            ok = True
            break

        except Exception as e:
            print(f"  attempt {attempt}/{MAX_PROJECT_RETRIES} failed: {type(e).__name__}: {e}")
            if attempt < MAX_PROJECT_RETRIES:
                time.sleep(2 * attempt)

    if not ok:
        if pid not in failed_ids:
            failed_ids.append(pid)
        WB_FAILS_PATH.write_text(json.dumps(failed_ids, indent=2), encoding="utf-8")
        print("  -> marked as failed")

    time.sleep(PROJECT_COOLDOWN_SEC)

WB_FAILS_PATH.write_text(json.dumps(failed_ids, indent=2), encoding="utf-8")

# final strict validation
if not final_df_wb.empty:
    final_df_wb = _ensure_output_schema(final_df_wb)
    final_df_wb = _sanitize_scores(final_df_wb, fill_ptrue_if_missing=not ENABLE_PTRUE)
    _assert_no_nan_output(final_df_wb, context="final_df_wb")

print("\nDone.")
print("Checkpoint:", WB_ROWS_PATH)
print("Rows:", len(final_df_wb))
print("Failed IDs:", failed_ids)

final_df_wb





In [57]:
final_df_wb.to_csv("uqlm_whitebox_Keine_Preisangabe_GPT20B.csv", index=False)

## Black‑Box Scorers (UQLM)

**1. Semantic Negentropy (Discrete Semantic Entropy)**  
$$\text{SemNeg} = 1 - \frac{-\sum_{c} p(c)\log p(c)}{\log(M+1)}$$

```python
    # responses (original + M samples, total M+1 items) are clustered by meaning using NLI (entailment)
    p_c = cluster_size / (M + 1)           # uniform 1/(M+1) per candidate, summed within cluster
    H   = -sum(p_c * log(p_c))             # semantic entropy
    semneg = 1 - H / log(M + 1)            # normalize by max entropy log(M+1) when all M+1 candidates differ
```

Cluster all sampled responses by semantic equivalence (NLI entailment). Use counts to form cluster probabilities, compute entropy, normalize by log(M+1), then invert to confidence. *Intuition: fewer distinct meanings → higher confidence.*

Note: Black‑box semantic entropy treats all sampled responses equally (count‑weighted). White‑box semantic entropy instead weights clusters by response probabilities from logprobs, so likely answers matter more and random low‑probability samples matter less. This usually gives a sharper confidence signal, but it can also hide uncertainty if the model is confidently wrong.

---

**2. Semantic Sets Confidence (Number of Semantic Sets)**  
$$\text{SSC} = 1 - \frac{S-1}{M-1}$$

```python
    S = num_semantic_sets                  # number of clusters
    ssc = 1 - (S - 1) / (M - 1)            # 1 if all samples agree, 0 if all differ
```

Convert the number of semantic clusters into a [0,1] confidence score. *Intuition: more clusters → less agreement.*

---

**3. Non‑Contradiction Probability**

$$
P(\text{contradiction}\mid y^*, y_i)
=\frac{1}{2}\Big(P_c(y^*,y_i) + P_c(y_i,y^*)\Big)
$$

$$
\text{NC} \;=\; \frac{1}{M}\sum_{i=1}^{M}\Big(1 - P(\text{contradiction}\mid y^*, y_i)\Big)
$$


```python
    # for each candidate y_i:
    nc_i = 1 - P(contradiction | y*, y_i)   # NLI both directions, averaged
    noncontradiction = mean(nc_i over samples)
```

Compute a non‑contradiction probability for each sampled response using NLI, then take the mean across samples. This gives the expected non‑contradiction level. *Intuition: fewer contradictions → higher confidence.*

---

**4. Entailment Probability**

$$
P(\text{entailment}\mid y^*, y_i)
=\frac{1}{2}\Big(P_e(y^*,y_i) + P_e(y_i,y^*)\Big)
$$

$$
\text{Ent} \;=\; \frac{1}{M}\sum_{i=1}^{M} P(\text{entailment}\mid y^*, y_i)
$$


```python
    # for each candidate y_i:
    ent_i = P(entailment | y*, y_i)         # NLI both directions, averaged
    entailment = mean(ent_i over samples)
```    

Use NLI to estimate entailment between original and samples, then average. *Intuition: more entailment → higher consistency.*

- Entailment checks if the sample supports the original answer.
- Non‑contradiction checks if the sample at least doesn’t conflict with it.
---

**5. Exact Match**

```python
    exact_match = mean([1 if y* == y_i else 0 for y_i in samples])
```

Exact string match rate between original response and samples. *Intuition: literal agreement → higher confidence.*

---

**6. Cosine Similarity (normalized)**

```python
    cos_i = cos(emb(y*), emb(y_i))          # embedding cosine similarity
    cos_i = 0.5 + cos_i / 2                 # normalize to [0,1]
    cosine_sim = mean(cos_i over samples)
```

Embedding similarity between original response and samples. *Intuition: semantic closeness → higher confidence.*

---

**7. BERTScore (F1)**

```python
    bert_f1 = mean(BERTScore_F1(y*, y_i) for y_i in samples)
```

BERTScore F1 between original response and samples. *Intuition: token‑level semantic overlap → higher confidence.*

Cosine similarity compares whole‑sentence embeddings (one vector per sentence).
BERTScore compares token‑level embeddings and aligns tokens across the two texts.

So:

Cosine similarity = coarse, fast, global meaning.
BERTScore = finer‑grained, captures local token‑level semantic matches.
BERTScore is usually more sensitive to partial overlap or paraphrases, but slower.


In [15]:
# =========================
# Black-Box UQ Pipeline (Improved, checkpointed, no test-mixing, Qwen/Groq/Mistral-safe)
# =========================
# Requires pre-existing notebook objects/functions:
# - PROJECT_IDS, TEST_MAP, gt_df, run_tests_for_project, get_simap_version

import os
import re
import sys
import json
import math
import time
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from requests import HTTPError

# Optional Google auth imports (needed for gemini_vertex backend)
try:
    import google.auth
    from google.auth.transport.requests import Request as GoogleAuthRequest
except Exception:
    google = None
    GoogleAuthRequest = None


# -------------------------
# 0) Paths + environment
# -------------------------
def _first_existing(paths):
    """Return first existing Path from list, else None."""
    for p in paths:
        if p.exists():
            return p
    return None


def _sanitize_tag(s: str) -> str:
    """Make run tag filesystem-safe."""
    return "".join(ch if ch.isalnum() or ch in ("_", "-", ".") else "_" for ch in s)


def _resolve_legal_service_root() -> Path:
    """Resolve legal_service directory from common notebook cwd layouts."""
    candidates = [
        Path("../intelliprocure-ai-legal/legal_service"),
        Path("intelliprocure-ai-legal/legal_service"),
        Path("../../intelliprocure-ai-legal/legal_service"),
    ]
    root = _first_existing(candidates)
    if root is None:
        raise FileNotFoundError("Could not locate intelliprocure-ai-legal/legal_service.")
    return root


def _resolve_ckpt_dir() -> Path:
    """Resolve checkpoint dir from common notebook cwd layouts."""
    candidates = [
        Path("_ckpt"),
        Path("uncertainty_estimation/_ckpt"),
        Path("uncertainty_estimation/uncertainty_estimation/_ckpt"),
    ]
    existing = _first_existing(candidates)
    if existing is not None:
        existing.mkdir(parents=True, exist_ok=True)
        return existing
    default = Path("uncertainty_estimation/_ckpt")
    default.mkdir(parents=True, exist_ok=True)
    return default


LEGAL_SERVICE_ROOT = _resolve_legal_service_root()
env_path = LEGAL_SERVICE_ROOT / ".env"
if env_path.exists():
    load_dotenv(env_path, override=True)
else:
    print("Warning: .env not found in legal_service; using current process env vars.")


# -------------------------
# 1) Backend config
# -------------------------
LLM_BACKEND = os.environ["LLM_BACKEND"].strip()

if LLM_BACKEND == "hf":
    LLM_BASE_URL = "https://router.huggingface.co/v1"
    LLM_API_KEY = os.environ["HF_TOKEN"]
    LLM_MODEL = os.environ["HF_MODEL"]

elif LLM_BACKEND == "openrouter":
    LLM_BASE_URL = "https://openrouter.ai/api/v1"
    LLM_API_KEY = os.environ["OPENROUTER_API_KEY"]
    LLM_MODEL = os.environ["OPENROUTER_MODEL"]

elif LLM_BACKEND == "together":
    LLM_BASE_URL = "https://api.together.xyz/v1"
    LLM_API_KEY = os.environ["TOGETHER_API_KEY"]
    LLM_MODEL = os.environ["TOGETHER_MODEL"]

elif LLM_BACKEND == "gemini_vertex":
    VERTEX_PROJECT_ID = os.environ["VERTEX_PROJECT_ID"]
    VERTEX_LOCATION = os.environ["VERTEX_LOCATION"]
    LLM_MODEL = os.environ["GEMINI_MODEL"]
    LLM_BASE_URL = f"https://{VERTEX_LOCATION}-aiplatform.googleapis.com/v1"
    LLM_API_KEY = None

else:
    raise ValueError(
        f"LLM_BACKEND must be 'hf', 'openrouter', 'together', or 'gemini_vertex', got: {LLM_BACKEND}"
    )

DATA_SERVICE_API_KEY = os.getenv("DATA_SERVICE_API_KEY", "")
DATA_SERVICE_HOST_URL = "http://localhost:8002"

print("LLM_BACKEND:", LLM_BACKEND)
print("LLM_MODEL:", LLM_MODEL)


# -------------------------
# 2) Run config
# -------------------------
NUM_SAMPLES = 5
SAMPLING_TEMPERATURE = 0.7
MAX_TOKENS = 4000
NLI_MAX_LENGTH = 4000

MAX_SAMPLE_TRIES_FACTOR = 3
MAX_PROJECT_RETRIES = 3

MAX_UPSTREAM_RETRIES = 10
BACKOFF_BASE = 2.0
BACKOFF_CAP = 180.0

MAX_CHAT_RETRIES = 8
CHAT_BACKOFF_BASE = 1.5
CHAT_BACKOFF_CAP = 120.0

PROJECT_COOLDOWN_SEC = 2.0


RUN_TAG = "Gerichtsferien_MISTRAL_24B"
ACTIVE_TEST_CODES = {"test_gerichtsferien"}

# For reasoning models (e.g., Qwen/Groq): remove <think> blocks before scoring
STRIP_THINK_BLOCKS = True


# -------------------------
# 3) Checkpoint paths
# -------------------------
CKPT_DIR = _resolve_ckpt_dir()
_TAG = _sanitize_tag(RUN_TAG)
BB_ROWS_PATH = CKPT_DIR / f"final_df_bb_partial_{_TAG}.parquet"
BB_FAILS_PATH = CKPT_DIR / f"failed_project_ids_bb_{_TAG}.json"


# -------------------------
# 4) Load UQLM
# -------------------------
sys.path.append("uqlm-main")
from uqlm.scorers.shortform.entropy import SemanticEntropy
from uqlm.black_box import CosineScorer, MatchScorer, ConsistencyScorer, BertScorer


# -------------------------
# 5) Core helpers
# -------------------------
def _strip_think_blocks(text: str) -> str:
    """Remove <think>...</think> blocks from model output."""
    if not isinstance(text, str):
        return ""
    return re.sub(r"<think>.*?</think>", " ", text, flags=re.IGNORECASE | re.DOTALL).strip()


def _clean_text_for_bb(text: str) -> str:
    """Normalize response text for black-box scoring."""
    t = text if isinstance(text, str) else str(text)
    t = t.strip()
    if STRIP_THINK_BLOCKS:
        t = _strip_think_blocks(t)

    # Defensive cleanup for some instruct-model wrappers.
    t = re.sub(r"</?s>|<unk>|\[/?INST\]", " ", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)

    return t.strip()


def extract_message_text(choice: dict) -> str:
    """Extract assistant text from OpenAI-like choice payload."""
    msg = (choice.get("message") or {})
    content = msg.get("content")

    if isinstance(content, str) and content.strip():
        return content.strip()

    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict):
                t = block.get("text")
                if isinstance(t, str) and t.strip():
                    parts.append(t.strip())
        if parts:
            return "\n".join(parts).strip()

    rc = msg.get("reasoning_content")
    if isinstance(rc, str) and rc.strip():
        return rc.strip()

    reasoning = msg.get("reasoning")
    if isinstance(reasoning, str) and reasoning.strip():
        return reasoning.strip()

    return ""


def _is_transient_http_error(e: HTTPError) -> bool:
    """Return True for retryable HTTP errors."""
    resp = getattr(e, "response", None)
    status = getattr(resp, "status_code", None)
    body = ""
    try:
        body = (resp.text or "").lower() if resp is not None else ""
    except Exception:
        pass

    if status in {429, 502, 503, 504}:
        return True
    if status == 500:
        markers = ["resource exhausted", "rate limit", "quota", "too many requests", "temporarily unavailable"]
        return any(m in body for m in markers)
    return False


def run_tests_for_project_with_backoff(project_id):
    """Call run_tests_for_project with exponential backoff on transient failures."""
    last_err = None
    for attempt in range(MAX_UPSTREAM_RETRIES):
        try:
            return run_tests_for_project(project_id)
        except HTTPError as e:
            last_err = e
            if not _is_transient_http_error(e):
                raise
            wait = min(BACKOFF_CAP, BACKOFF_BASE * (2 ** attempt)) + np.random.uniform(0, 0.7)
            print(f"run_tests_for_project transient error; retry {attempt+1}/{MAX_UPSTREAM_RETRIES} in {wait:.1f}s")
            time.sleep(wait)
    raise last_err


# -------------------------
# 6) Backend chat helpers
# -------------------------
_VERTEX_CREDS = None

def _vertex_access_token(force_refresh: bool = False) -> str:
    """Get/refresh ADC token for Vertex API."""
    global _VERTEX_CREDS
    if google is None or GoogleAuthRequest is None:
        raise RuntimeError("google-auth is required for gemini_vertex backend.")

    if _VERTEX_CREDS is None:
        _VERTEX_CREDS, _ = google.auth.default(
            scopes=["https://www.googleapis.com/auth/cloud-platform"]
        )

    if force_refresh or (not _VERTEX_CREDS.valid) or (_VERTEX_CREDS.token is None):
        _VERTEX_CREDS.refresh(GoogleAuthRequest())

    return _VERTEX_CREDS.token


def _messages_to_gemini_payload(messages, temperature, max_tokens):
    """Convert OpenAI-style messages to Vertex Gemini payload."""
    contents = []
    system_texts = []

    for m in messages:
        role = (m.get("role") or "").lower()
        content = m.get("content", "")

        if isinstance(content, list):
            text = "\n".join(
                p.get("text", "").strip()
                for p in content
                if isinstance(p, dict) and isinstance(p.get("text"), str) and p.get("text").strip()
            ).strip()
        else:
            text = str(content).strip()

        if not text:
            continue

        if role == "system":
            system_texts.append(text)
        else:
            gem_role = "model" if role == "assistant" else "user"
            contents.append({"role": gem_role, "parts": [{"text": text}]})

    payload = {
        "contents": contents if contents else [{"role": "user", "parts": [{"text": ""}]}],
        "generationConfig": {
            "temperature": float(temperature),
            "maxOutputTokens": int(max_tokens),
        },
    }

    if system_texts:
        payload["systemInstruction"] = {
            "role": "system",
            "parts": [{"text": "\n\n".join(system_texts)}],
        }

    return payload


def _vertex_chat(messages, temperature=0.7, max_tokens=MAX_TOKENS):
    """Call Vertex Gemini and normalize to OpenAI-like response shape."""
    url = (
        f"{LLM_BASE_URL}/projects/{VERTEX_PROJECT_ID}/locations/{VERTEX_LOCATION}/"
        f"publishers/google/models/{LLM_MODEL}:generateContent"
    )
    body = _messages_to_gemini_payload(messages, temperature=temperature, max_tokens=max_tokens)

    token = _vertex_access_token(force_refresh=False)
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    r = requests.post(url, json=body, headers=headers, timeout=120)

    if r.status_code in (401, 403):
        token = _vertex_access_token(force_refresh=True)
        headers["Authorization"] = f"Bearer {token}"
        r = requests.post(url, json=body, headers=headers, timeout=120)

    if r.status_code >= 400:
        raise HTTPError(f"{r.status_code} {r.reason} - {r.text[:1200]}", response=r)

    j = r.json()
    cand = (j.get("candidates") or [{}])[0]
    parts = ((cand.get("content") or {}).get("parts") or [])
    text = "".join(p.get("text", "") for p in parts if isinstance(p, dict)).strip()

    return {"choices": [{"message": {"role": "assistant", "content": text}}], "raw": j}


def llm_chat(messages, temperature=0.7, max_tokens=MAX_TOKENS):
    """Dispatch chat call to configured backend."""
    if LLM_BACKEND == "gemini_vertex":
        return _vertex_chat(messages, temperature=temperature, max_tokens=max_tokens)

    payload = {
        "model": LLM_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    if LLM_BACKEND == "openrouter":
        payload["provider"] = {"require_parameters": False}

    headers = {"Authorization": f"Bearer {LLM_API_KEY}"}

    # Router-specific headers
    if LLM_BACKEND in {"hf", "openrouter"}:
        headers["HTTP-Referer"] = "https://intelliprocure.ch"
        headers["X-Title"] = "UQLM Black-Box Eval"
    elif LLM_BACKEND in {"together", "together_ai"}:
        headers["Content-Type"] = "application/json"

    r = requests.post(f"{LLM_BASE_URL}/chat/completions", json=payload, headers=headers, timeout=120)
    if r.status_code >= 400:
        raise HTTPError(f"{r.status_code} {r.reason} - {r.text[:1200]}", response=r)

    return r.json()


def llm_chat_with_backoff(messages, temperature=0.7, max_tokens=MAX_TOKENS):
    """Call llm_chat with exponential backoff for transient failures."""
    last_err = None
    for attempt in range(MAX_CHAT_RETRIES):
        try:
            return llm_chat(messages, temperature=temperature, max_tokens=max_tokens)
        except HTTPError as e:
            last_err = e
            if not _is_transient_http_error(e):
                raise
            wait = min(CHAT_BACKOFF_CAP, CHAT_BACKOFF_BASE * (2 ** attempt)) + np.random.uniform(0, 0.6)
            print(f"llm_chat transient error; retry {attempt+1}/{MAX_CHAT_RETRIES} in {wait:.1f}s")
            time.sleep(wait)
    raise last_err


# -------------------------
# 7) Prompt/data helpers
# -------------------------
def build_messages(check_code, project_summary):
    """Rebuild legal_service prompt/messages for a given check code."""
    checks_cfg = json.loads(
        (LEGAL_SERVICE_ROOT / "test_definitions/checks.json").read_text(encoding="utf-8")
    )
    prompts_dir = LEGAL_SERVICE_ROOT / "test_definitions/prompts_checks"
    system_prompt = (LEGAL_SERVICE_ROOT / "test_definitions/system_prompt.txt").read_text(encoding="utf-8").strip()

    cfg = checks_cfg[check_code]
    required_fields = cfg["required_fields"]
    prompt_file = cfg["prompt_file"]

    def format_project_data(project_data):
        """Format required fields into check prompt template."""
        formatted = ""
        for k, v in project_data.items():
            if v is None or v == "":
                v = "No answer"
            elif not isinstance(v, str):
                v = str(v)
            formatted += f"**{k}:**\n{v}\n\n"
        return formatted

    project_data = {f: project_summary.get(f, "") for f in required_fields}
    prompt = (prompts_dir / prompt_file).read_text(encoding="utf-8").replace(
        "{PROJECT_DATA}", format_project_data(project_data)
    )

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]


def fetch_project_summary(project_id, simap_version):
    """Fetch project summary from data_service."""
    url = f"{DATA_SERVICE_HOST_URL}/api/v1/projects/{project_id}?simap_version={simap_version}"
    headers = {"X-API-KEY": DATA_SERVICE_API_KEY} if DATA_SERVICE_API_KEY else {}
    r = requests.get(url, headers=headers, timeout=60)
    r.raise_for_status()
    return r.json()["data"]["attributes"]["summary"]


# -------------------------
# 8) Per-project execution
# -------------------------
def run_one_project_bb(project_id):
    """Run legal checks + black-box UQ scoring for one project."""
    resp = run_tests_for_project_with_backoff(project_id)

    gt_row = gt_df.loc[gt_df["GT table"] == project_id]
    if gt_row.empty:
        print("No GT row, skipping", project_id)
        return None
    gt_row = gt_row.iloc[0]

    by_code = {r["test_code"]: r for r in resp["data"]["results"]}
    gt_by_test = {}
    for display_name, test_code in TEST_MAP.items():
        gt = bool(gt_row[display_name])
        pred = bool(by_code.get(test_code, {}).get("violation_detected"))
        gt_by_test[test_code] = (gt == pred)

    records = []
    for r in resp["data"]["results"]:
        if ACTIVE_TEST_CODES and r["test_code"] not in ACTIVE_TEST_CODES:
            continue

        label = gt_by_test.get(r["test_code"])
        for c in r.get("execution_info", {}).get("checks_executed", []):
            records.append(
                {
                    "project_id": project_id,
                    "test_code": r["test_code"],
                    "check_code": c["check_code"],
                    "response": _clean_text_for_bb(c.get("llm_answer", "")),
                    "label": int(label) if label is not None else None,
                }
            )

    if not records:
        return pd.DataFrame()

    simap_version = get_simap_version(project_id)
    project_summary = fetch_project_summary(project_id, simap_version)

    sampled_responses = []
    valid_records = []

    for rec in records:
        msgs = build_messages(rec["check_code"], project_summary)

        sample_texts = []
        tries = 0
        max_tries = NUM_SAMPLES * MAX_SAMPLE_TRIES_FACTOR

        while len(sample_texts) < NUM_SAMPLES and tries < max_tries:
            tries += 1
            try:
                data = llm_chat_with_backoff(
                    msgs,
                    temperature=SAMPLING_TEMPERATURE,
                    max_tokens=MAX_TOKENS,
                )
            except HTTPError:
                continue

            choice = (data.get("choices") or [{}])[0]
            text = _clean_text_for_bb(extract_message_text(choice))
            if text:
                sample_texts.append(text)

        if len(sample_texts) != NUM_SAMPLES:
            continue

        sampled_responses.append(sample_texts)
        valid_records.append(rec)

    records = valid_records
    if not records:
        return pd.DataFrame()

    responses = [rec["response"] for rec in records]
    M = NUM_SAMPLES

    se = SemanticEntropy(
        llm=None,
        nli_model_name="microsoft/deberta-large-mnli",
        max_length=NLI_MAX_LENGTH,
        use_best=False,
    )
    se_result = se.score(
        responses=responses,
        sampled_responses=sampled_responses,
        show_progress_bars=False,
    )

    discrete_entropy = se_result.data["discrete_entropy_values"]
    num_sem_sets = se_result.data["num_semantic_sets"]

    semantic_negentropy = [1 - (e / math.log(M + 1)) for e in discrete_entropy]
    semantic_sets_confidence = [1 - (s - 1) / max(M - 1, 1) for s in num_sem_sets]

    cons = ConsistencyScorer(
        nli_model_name="microsoft/deberta-large-mnli",
        max_length=NLI_MAX_LENGTH,
        use_best=False,
        scorers=["noncontradiction", "entailment"],
    )
    cons_scores = cons.evaluate(
        responses=responses,
        sampled_responses=sampled_responses,
        progress_bar=None,
    )

    exact_match = MatchScorer().evaluate(
        responses=responses,
        sampled_responses=sampled_responses,
        progress_bar=None,
    )
    cosine_sim = CosineScorer(transformer="sentence-transformers/all-MiniLM-L6-v2").evaluate(
        responses=responses,
        sampled_responses=sampled_responses,
        progress_bar=None,
    )

    try:
        bert_score = BertScorer(device=None).evaluate(
            responses=responses,
            sampled_responses=sampled_responses,
            progress_bar=None,
        )
    except Exception as e:
        print("BERTScore failed:", e)
        bert_score = [0.0] * len(responses)

    scores_df = pd.DataFrame(
        {
            "semantic_negentropy_bb": semantic_negentropy,
            "semantic_sets_confidence": semantic_sets_confidence,
            "noncontradiction": cons_scores["noncontradiction"],
            "entailment": cons_scores["entailment"],
            "exact_match": exact_match,
            "cosine_sim": cosine_sim,
            "bert_score": bert_score,
        }
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    pred_by_test = {r["test_code"]: r.get("violation_detected") for r in resp["data"]["results"]}
    gt_val_by_test = {test_code: bool(gt_row[display_name]) for display_name, test_code in TEST_MAP.items()}

    records_df = pd.DataFrame(records)
    records_df["gt"] = records_df["test_code"].map(gt_val_by_test)
    records_df["pred"] = records_df["test_code"].map(pred_by_test)
    records_df["match"] = records_df["gt"] == records_df["pred"]

    merged = pd.concat([records_df.reset_index(drop=True), scores_df.reset_index(drop=True)], axis=1)
    merged = merged[
        ["project_id", "test_code", "check_code", "gt", "pred", "match", "response"] + list(scores_df.columns)
    ]
    return merged


# -------------------------
# 9) Recovery + run-all
# -------------------------
if BB_ROWS_PATH.exists():
    try:
        final_df_bb = pd.read_parquet(BB_ROWS_PATH)
    except Exception:
        final_df_bb = pd.DataFrame()
else:
    final_df_bb = pd.DataFrame()

if ACTIVE_TEST_CODES and not final_df_bb.empty and "test_code" in final_df_bb.columns:
    final_df_bb = final_df_bb[final_df_bb["test_code"].isin(ACTIVE_TEST_CODES)].copy()

done_ids = set(final_df_bb["project_id"].unique()) if (not final_df_bb.empty and "project_id" in final_df_bb.columns) else set()
pending_ids = [pid for pid in PROJECT_IDS if pid not in done_ids]

if BB_FAILS_PATH.exists():
    try:
        failed_ids = json.loads(BB_FAILS_PATH.read_text(encoding="utf-8"))
    except Exception:
        failed_ids = []
else:
    failed_ids = []

failed_ids = [pid for pid in failed_ids if pid in PROJECT_IDS]

print(f"RUN_TAG: {_TAG}")
print(f"ACTIVE_TEST_CODES: {ACTIVE_TEST_CODES}")
print(f"Recovered rows: {len(final_df_bb)}")
print(f"Done IDs: {len(done_ids)} | Pending IDs: {len(pending_ids)}")

for pid in pending_ids:
    print("\nRunning", pid)
    ok = False

    for attempt in range(1, MAX_PROJECT_RETRIES + 1):
        try:
            out = run_one_project_bb(pid)

            if out is not None and not out.empty:
                out = out.copy()
                if ACTIVE_TEST_CODES:
                    out = out[out["test_code"].isin(ACTIVE_TEST_CODES)].copy()

                if not out.empty:
                    if not final_df_bb.empty:
                        if ACTIVE_TEST_CODES:
                            mask = (final_df_bb["project_id"] == pid) & (final_df_bb["test_code"].isin(ACTIVE_TEST_CODES))
                        else:
                            mask = (final_df_bb["project_id"] == pid)
                        final_df_bb = final_df_bb.loc[~mask].copy()

                    final_df_bb = pd.concat([final_df_bb, out], ignore_index=True)
                    final_df_bb = final_df_bb.drop_duplicates(
                        subset=["project_id", "test_code", "check_code"], keep="last"
                    )
                    final_df_bb.to_parquet(BB_ROWS_PATH, index=False)
                    print("  saved rows:", len(final_df_bb))
                else:
                    print("  output had no active tests after filtering")
            else:
                print("  no rows returned")

            if pid in failed_ids:
                failed_ids = [x for x in failed_ids if x != pid]
                BB_FAILS_PATH.write_text(json.dumps(failed_ids, indent=2), encoding="utf-8")

            ok = True
            break

        except Exception as e:
            print(f"  attempt {attempt}/{MAX_PROJECT_RETRIES} failed: {type(e).__name__}: {e}")
            if attempt < MAX_PROJECT_RETRIES:
                time.sleep(2 * attempt)

    if not ok:
        if pid not in failed_ids:
            failed_ids.append(pid)
        BB_FAILS_PATH.write_text(json.dumps(failed_ids, indent=2), encoding="utf-8")
        print("  -> marked as failed")

    time.sleep(PROJECT_COOLDOWN_SEC)

BB_FAILS_PATH.write_text(json.dumps(failed_ids, indent=2), encoding="utf-8")

print("\nDone.")
print("Checkpoint:", BB_ROWS_PATH)
print("Rows:", len(final_df_bb))
print("Failed IDs:", failed_ids)

final_df_bb


LLM_BACKEND: together
LLM_MODEL: mistralai/Mistral-Small-24B-Instruct-2501
RUN_TAG: Gerichtsferien_MISTRAL_24B
ACTIVE_TEST_CODES: {'test_gerichtsferien'}
Recovered rows: 0
Done IDs: 0 | Pending IDs: 13

Running 191846
  saved rows: 1

Running 169402
  saved rows: 2

Running 189879
  saved rows: 3

Running 235296
  saved rows: 4

Running 258830
  saved rows: 5

Running 190944
  saved rows: 6

Running 243602
  saved rows: 7

Running 205270
  saved rows: 8

Running 229691
  saved rows: 9

Running 221633
  saved rows: 10

Running 9583
  saved rows: 11

Running 196565
  saved rows: 12

Running 192494
  saved rows: 13

Done.
Checkpoint: uncertainty_estimation\_ckpt\final_df_bb_partial_Gerichtsferien_MISTRAL_24B.parquet
Rows: 13
Failed IDs: []


,project_id,test_code,check_code,gt,pred,match,response,semantic_negentropy_bb,semantic_sets_confidence,noncontradiction,entailment,exact_match,cosine_sim,bert_score
0,191846,test_gerichtsferien,court_holiday,False,False,True,"Um die Aufgabe zu lösen, analysiere ich das Fe...",1.0,1.0,0.818966,0.715821,0.0,0.929002,0.906853
1,169402,test_gerichtsferien,court_holiday,False,False,True,"Um die Aufgabe zu lösen, analysieren wir das F...",1.0,1.0,0.877818,0.806830,0.0,0.943533,0.925702
2,189879,test_gerichtsferien,court_holiday,False,False,True,"Um die Aufgabe zu lösen, analysiere ich das Fe...",1.0,1.0,0.869918,0.750509,0.0,0.941267,0.919725
3,235296,test_gerichtsferien,court_holiday,False,False,True,"Um die Aufgabe zu lösen, analysieren wir das F...",1.0,1.0,0.853308,0.776178,0.0,0.895378,0.872291
4,258830,test_gerichtsferien,court_holiday,False,False,True,"Um die Aufgabe zu lösen, analysieren wir das F...",1.0,1.0,0.840089,0.765236,0.0,0.928121,0.932697
5,190944,test_gerichtsferien,court_holiday,False,False,True,"Um die Aufgabe zu lösen, analysiere ich das Fe...",1.0,1.0,0.829120,0.729894,0.0,0.927890,0.896329
6,243602,test_gerichtsferien,court_holiday,False,False,True,"Um die Aufgabe zu lösen, analysiere ich das Fe...",1.0,1.0,0.854249,0.767140,0.0,0.926018,0.914910
7,205270,test_gerichtsferien,court_holiday,False,False,True,"Um die Aufgabe zu lösen, analysieren wir das F...",1.0,1.0,0.864396,0.788411,0.0,0.899398,0.885284
8,229691,test_gerichtsferien,court_holiday,False,False,True,"Um die Aufgabe zu lösen, analysiere ich das Fe...",1.0,1.0,0.844321,0.746764,0.0,0.958814,0.929228
9,221633,test_gerichtsferien,court_holiday,False,False,True,"Um die Aufgabe zu lösen, analysiere ich das Fe...",1.0,1.0,0.883132,0.801812,0.0,0.956319,0.944791


In [16]:
final_df_bb.to_csv("uqlm_blackbox_Gerichtsferien_MIISTRAL_24B.csv", index=False)

## Judge-Based Scorers (UQLM)

**1. Categorical LLM-as-a-Judge**
$$J_{\text{cat}}(y) = \begin{cases} 0 & \text{incorrect} \\ 0.5 & \text{uncertain (only in true\_false\_uncertain)} \\ 1 & \text{correct} \end{cases}$$
```python
# judge outputs text label, parser maps keywords -> score
if template == "true_false_uncertain": score in {0, 0.5, 1}
if template == "true_false":           score in {0, 1}
```
Judge classifies answer correctness into categories. *Intuition: direct verdict-style judging — with or without a middle "uncertain" option.*

---

**2. Continuous LLM-as-a-Judge**
$$J_{\text{cont}}(y) = \frac{s}{100}, \quad s \in [0, 100]$$
```python
# judge returns 0..100 confidence, UQLM normalizes to 0..1
score = float(extracted_digits) / 100
```
Judge gives a numeric confidence score instead of a class, then UQLM normalizes it to [0,1]. *Intuition: finer-grained correctness confidence.*

---

**3. Likert Scale LLM-as-a-Judge**
$$J_{\text{likert}}(y) = \frac{\ell - 1}{4}, \quad \ell \in \{1, 2, 3, 4, 5\}$$
```python
# 1..5 mapped to 0, .25, .5, .75, 1
score = (int(likert_response) - 1) / 4
```
Judge rates correctness on a 5-point rubric, then score is normalized to [0,1]. *Intuition: structured human-like grading with moderate granularity.*

---

**4. Panel of LLM Judges**
$$\text{Panel}_{\text{avg}}(y) = \frac{1}{n}\sum_{k=1}^{n} J_k(y), \quad \text{Panel}_{\text{max}}(y) = \max_k J_k(y), \quad \text{Panel}_{\text{min}}(y) = \min_k J_k(y), \quad \text{Panel}_{\text{med}}(y) = \text{median}_k\, J_k(y)$$
```python
# multiple judges score same (question, answer), then aggregate
avg    = mean(scores_from_all_judges)
maxv   = max(scores_from_all_judges)
minv   = min(scores_from_all_judges)
median = median(scores_from_all_judges)
```
Multiple judges score the same response, then their scores are aggregated by mean, max, min, or median. *Intuition: reduce single-judge noise and bias by combining opinions from several judges.*


In [63]:
# =========================
# Judge Pipeline (manual file names)
# =========================
# Set exactly the 2 files you want each run:
# - one blackbox CSV
# - one whitebox CSV

import os
import re
import json
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from dotenv import load_dotenv
from requests import HTTPError
from IPython.display import display

# Optional for Vertex
try:
    import google.auth
    from google.auth.transport.requests import Request as GoogleAuthRequest
except Exception:
    google = None
    GoogleAuthRequest = None


# -------------------------
# 0) Manual inputs
# -------------------------
BLACKBOX_FILE = "uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv"
WHITEBOX_FILE = "uqlm_whitebox_Verzerrung_Bewertung_GPT20B.csv"

ONLY_MATCH_FALSE = False
MAX_ROWS = None

CSV_DIR = Path(".")
LEGAL_SERVICE_ROOT = Path("../intelliprocure-ai-legal/legal_service")

MAX_TOKENS_JUDGE = 1024
JUDGE_MAX_ATTEMPTS = 3
JUDGE_TEMPERATURE = 0.0

JUDGE_SYSTEM_PROMPT = (
    "Du bist eine Expertin für die Bewertung von Antworten in juristischen Beschaffungsprüfungen. "
    "Bewerte nur die Korrektheit der vorgeschlagenen Antwort zur Frage."
)
JUDGE_INSTRUCTION = (
    "Bewerte nur die Korrektheit der vorgeschlagenen Antwort.\n"
    "Gib GENAU EINE Zahl zurück: 0 oder 0.5 oder 1\n"
    "Bedeutung: 0=Inkorrekt, 0.5=Unsicher, 1=Korrekt.\n"
    "Keine Begründung. Kein weiterer Text."
)


load_dotenv(LEGAL_SERVICE_ROOT / ".env", override=True)
DATA_SERVICE_HOST_URL = os.getenv("DATA_SERVICE_HOST_URL", "http://localhost:8002")
DATA_SERVICE_API_KEY = os.getenv("DATA_SERVICE_API_KEY", "")


# -------------------------
# 1) Judge backend from env
# -------------------------
LLM_BACKEND = os.environ["LLM_BACKEND"].strip()

if LLM_BACKEND == "hf":
    LLM_BASE_URL = "https://router.huggingface.co/v1"
    LLM_API_KEY = os.environ["HF_TOKEN"]
    LLM_MODEL = os.environ["HF_MODEL"]

elif LLM_BACKEND == "openrouter":
    LLM_BASE_URL = "https://openrouter.ai/api/v1"
    LLM_API_KEY = os.environ["OPENROUTER_API_KEY"]
    LLM_MODEL = os.environ["OPENROUTER_MODEL"]

elif LLM_BACKEND == "together":
    LLM_BASE_URL = "https://api.together.xyz/v1"
    LLM_API_KEY = os.environ["TOGETHER_API_KEY"]
    LLM_MODEL = os.environ["TOGETHER_MODEL"]

elif LLM_BACKEND == "gemini_vertex":
    VERTEX_PROJECT_ID = os.environ["VERTEX_PROJECT_ID"]
    VERTEX_LOCATION = os.environ["VERTEX_LOCATION"]
    LLM_MODEL = os.environ["GEMINI_MODEL"]
    LLM_BASE_URL = f"https://{VERTEX_LOCATION}-aiplatform.googleapis.com/v1"
    LLM_API_KEY = None

else:
    raise ValueError(
        f"LLM_BACKEND must be 'hf', 'openrouter', 'together', or 'gemini_vertex', got: {LLM_BACKEND}"
    )

print("Judge backend:", LLM_BACKEND)
print("Judge model:  ", LLM_MODEL)


# -------------------------
# 2) Common helpers
# -------------------------
def parse_bool_like(v: Any) -> Optional[bool]:
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)) and not pd.isna(v):
        if v in (0, 0.0):
            return False
        if v in (1, 1.0):
            return True
    if isinstance(v, str):
        t = v.strip().lower()
        if t in {"true", "t", "1", "yes", "y"}:
            return True
        if t in {"false", "f", "0", "no", "n"}:
            return False
    return None

def load_manual_file(path: Path, source_kind: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    needed = {"project_id", "test_code", "check_code", "response"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"{path.name} missing required columns: {missing}")

    for c in ["gt", "pred", "match"]:
        if c not in df.columns:
            df[c] = np.nan

    out = df.copy()
    out["match_bool"] = out["match"].apply(parse_bool_like)

    if ONLY_MATCH_FALSE:
        out = out[out["match_bool"] == False].copy()

    out["source"] = source_kind
    out["source_file"] = path.name

    out["project_id"] = pd.to_numeric(out["project_id"], errors="coerce")
    out = out.dropna(subset=["project_id"]).copy()
    out["project_id"] = out["project_id"].astype(int)

    cols = ["source", "source_file", "project_id", "test_code", "check_code", "gt", "pred", "match", "response"]
    out = out[cols]

    if MAX_ROWS is not None:
        out = out.head(int(MAX_ROWS)).copy()

    return out


# -------------------------
# 3) Load exactly those 2 files
# -------------------------
bb_path = CSV_DIR / BLACKBOX_FILE
wb_path = CSV_DIR / WHITEBOX_FILE

if not bb_path.exists():
    raise FileNotFoundError(f"Blackbox file not found: {bb_path.resolve()}")
if not wb_path.exists():
    raise FileNotFoundError(f"Whitebox file not found: {wb_path.resolve()}")

bb_df = load_manual_file(bb_path, "blackbox")
wb_df = load_manual_file(wb_path, "whitebox")

base_df = pd.concat([bb_df, wb_df], ignore_index=True)
print("Rows to judge:", len(base_df))
display(base_df.head(20))


# -------------------------
# 4) Prompt rebuild + project summaries
# -------------------------
checks_cfg = json.loads((LEGAL_SERVICE_ROOT / "test_definitions/checks.json").read_text(encoding="utf-8"))
prompts_dir = LEGAL_SERVICE_ROOT / "test_definitions/prompts_checks"

def build_version_map() -> Dict[int, List[str]]:
    r = requests.get(f"{DATA_SERVICE_HOST_URL}/projects", timeout=60)
    r.raise_for_status()
    projects = r.json()["data"]["projects"]
    vm: Dict[int, List[str]] = {}
    for p in projects:
        vm.setdefault(int(p["project_id"]), []).append(p["simap_version"])
    return vm

def get_simap_version(project_id: int, version_map: Dict[int, List[str]]) -> str:
    vals = version_map.get(project_id, [])
    if not vals:
        return "simap"
    return "simap_v2" if "simap_v2" in vals else vals[0]

def fetch_project_summary(project_id: int, simap_version: str) -> Dict[str, Any]:
    url = f"{DATA_SERVICE_HOST_URL}/api/v1/projects/{project_id}?simap_version={simap_version}"
    headers = {"X-API-KEY": DATA_SERVICE_API_KEY} if DATA_SERVICE_API_KEY else {}
    r = requests.get(url, headers=headers, timeout=60)
    r.raise_for_status()
    return r.json()["data"]["attributes"]["summary"]

def build_check_question(check_code: str, project_summary: Dict[str, Any]) -> str:
    cfg = checks_cfg[check_code]
    required_fields = cfg["required_fields"]
    prompt_file = cfg["prompt_file"]

    formatted = ""
    for k in required_fields:
        v = project_summary.get(k, "")
        if v is None or v == "":
            v = "No answer"
        elif not isinstance(v, str):
            v = str(v)
        formatted += f"**{k}:**\n{v}\n\n"

    template = (prompts_dir / prompt_file).read_text(encoding="utf-8")
    return template.replace("{PROJECT_DATA}", formatted)

version_map = build_version_map()


# -------------------------
# 5) Judge call helpers
# -------------------------
_VERTEX_CREDS = None

def _vertex_access_token(force_refresh: bool = False) -> str:
    global _VERTEX_CREDS
    if google is None or GoogleAuthRequest is None:
        raise RuntimeError("google-auth is required for gemini_vertex backend.")
    if _VERTEX_CREDS is None:
        _VERTEX_CREDS, _ = google.auth.default(
            scopes=["https://www.googleapis.com/auth/cloud-platform"]
        )
    if force_refresh or (not _VERTEX_CREDS.valid) or (_VERTEX_CREDS.token is None):
        _VERTEX_CREDS.refresh(GoogleAuthRequest())
    return _VERTEX_CREDS.token

def _messages_to_gemini_payload(messages: List[Dict[str, str]], temperature: float, max_tokens: int) -> Dict[str, Any]:
    contents = []
    system_texts = []

    for m in messages:
        role = (m.get("role") or "").lower()
        text = str(m.get("content", "")).strip()
        if not text:
            continue
        if role == "system":
            system_texts.append(text)
        else:
            gem_role = "model" if role == "assistant" else "user"
            contents.append({"role": gem_role, "parts": [{"text": text}]})

    payload = {
        "contents": contents if contents else [{"role": "user", "parts": [{"text": ""}]}],
        "generationConfig": {"temperature": float(temperature), "maxOutputTokens": int(max_tokens)},
    }
    if system_texts:
        payload["systemInstruction"] = {"role": "system", "parts": [{"text": "\n\n".join(system_texts)}]}
    return payload

def extract_message_text(data: Dict[str, Any]) -> str:
    choice = (data.get("choices") or [{}])[0]
    msg = choice.get("message") or {}
    content = msg.get("content", "")

    if isinstance(content, str):
        return content.strip()

    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict):
                t = block.get("text")
                if isinstance(t, str) and t.strip():
                    parts.append(t.strip())
        if parts:
            return "\n".join(parts).strip()

    reasoning = msg.get("reasoning")
    if isinstance(reasoning, str) and reasoning.strip():
        return reasoning.strip()

    return ""

def judge_chat(messages: List[Dict[str, str]], max_tokens: int, temperature: float) -> Dict[str, Any]:
    if LLM_BACKEND == "gemini_vertex":
        url = (
            f"{LLM_BASE_URL}/projects/{VERTEX_PROJECT_ID}/locations/{VERTEX_LOCATION}/"
            f"publishers/google/models/{LLM_MODEL}:generateContent"
        )
        body = _messages_to_gemini_payload(messages, temperature=temperature, max_tokens=max_tokens)
        token = _vertex_access_token(force_refresh=False)
        headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
        r = requests.post(url, json=body, headers=headers, timeout=120)

        if r.status_code in (401, 403):
            token = _vertex_access_token(force_refresh=True)
            headers["Authorization"] = f"Bearer {token}"
            r = requests.post(url, json=body, headers=headers, timeout=120)

        r.raise_for_status()
        j = r.json()
        cand = (j.get("candidates") or [{}])[0]
        parts = ((cand.get("content") or {}).get("parts") or [])
        text = "".join(p.get("text", "") for p in parts if isinstance(p, dict)).strip()
        return {"choices": [{"message": {"role": "assistant", "content": text}}], "raw": j}

    payload = {
        "model": LLM_MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    headers = {"Authorization": f"Bearer {LLM_API_KEY}"}
    if LLM_BACKEND in {"hf", "openrouter"}:
        headers["HTTP-Referer"] = "https://intelliprocure.ch"
        headers["X-Title"] = "UQLM Judge Eval"
    elif LLM_BACKEND == "together":
        headers["Content-Type"] = "application/json"

    r = requests.post(f"{LLM_BASE_URL}/chat/completions", json=payload, headers=headers, timeout=120)
    r.raise_for_status()
    return r.json()

def parse_judge_numeric_score(raw_text: str) -> Optional[float]:
    txt = (raw_text or "").strip()
    if txt in {"0", "0.0"}:
        return 0.0
    if txt in {"0.5", ".5"}:
        return 0.5
    if txt in {"1", "1.0"}:
        return 1.0

    m = re.search(r"\b(0(?:\.5)?|1(?:\.0)?)\b", txt)
    if m:
        v = float(m.group(1))
        if v in (0.0, 0.5, 1.0):
            return v
    return None

def judge_score_with_retry(question_text: str, proposed_answer: str) -> Tuple[float, str]:
    base_prompt = (
        f"Frage: {question_text}\n\n"
        f"Vorgeschlagene Antwort: {proposed_answer}\n\n"
        f"{JUDGE_INSTRUCTION}"
    )
    last_raw = ""

    for attempt in range(1, JUDGE_MAX_ATTEMPTS + 1):
        user_prompt = base_prompt if attempt == 1 else (
            f"{base_prompt}\n\nUngültige Ausgabe. Antworte jetzt nur mit einer Zahl: 0 oder 0.5 oder 1."
        )

        try:
            data = judge_chat(
                messages=[
                    {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                max_tokens=MAX_TOKENS_JUDGE,
                temperature=JUDGE_TEMPERATURE,
            )
            raw = extract_message_text(data)
            last_raw = raw
            parsed = parse_judge_numeric_score(raw)
            if parsed is not None:
                return float(parsed), raw

        except HTTPError as e:
            last_raw = e.response.text[:500] if (e.response is not None and e.response.text) else str(e)
        except Exception as e:
            last_raw = str(e)

    return np.nan, last_raw


# -------------------------
# 6) Run judge + merge
# -------------------------
summary_cache: Dict[int, Dict[str, Any]] = {}
question_cache: Dict[Tuple[int, str], str] = {}

def get_question_text(project_id: int, check_code: str) -> str:
    key = (project_id, check_code)
    if key in question_cache:
        return question_cache[key]

    if project_id not in summary_cache:
        simap_version = get_simap_version(project_id, version_map)
        summary_cache[project_id] = fetch_project_summary(project_id, simap_version)

    q = build_check_question(check_code, summary_cache[project_id])
    question_cache[key] = q
    return q

judge_rows = []
N = len(base_df)

for i, row in base_df.reset_index(drop=True).iterrows():
    if i == 0 or (i + 1) % 25 == 0 or i == N - 1:
        print(f"Judging {i+1}/{N}")

    pid = int(row["project_id"])
    cc = str(row["check_code"])
    answer = "" if pd.isna(row["response"]) else str(row["response"])

    q = get_question_text(pid, cc)
    s, raw = judge_score_with_retry(q, answer)

    judge_rows.append({
        "judge_backend": LLM_BACKEND,
        "judge_model": LLM_MODEL,
        "judge_score": s,
        "judge_raw": raw,
    })

judge_df = pd.DataFrame(judge_rows)
final_judge_table = pd.concat([base_df.reset_index(drop=True), judge_df], axis=1)
final_judge_table = final_judge_table.sort_values(["source", "test_code", "project_id", "check_code"]).reset_index(drop=True)

print("\nFinal merged table:")
display(final_judge_table)



Judge backend: gemini_vertex
Judge model:   gemini-2.5-flash
Rows to judge: 26


,source,source_file,project_id,test_code,check_code,gt,pred,match,response
0,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,191846,test_verzerrung_bewertung,min_score,False,False,True,Zuerst lese ich den Text im Feld „award_criter...
1,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,169402,test_verzerrung_bewertung,min_score,False,False,True,"Zunächst prüfen wir, ob im Text explizit eine ..."
2,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,189879,test_verzerrung_bewertung,min_score,False,False,True,Zuerst lese ich den Text im Feld „award_criter...
3,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,235296,test_verzerrung_bewertung,min_score,False,False,True,Zunächst wird der Text des Feldes „award_crite...
4,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,258830,test_verzerrung_bewertung,min_score,False,False,True,Zuerst lese ich den Text im Feld „award_criter...
5,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,190944,test_verzerrung_bewertung,min_score,False,False,True,Zunächst lese ich den gesamten Text des Feldes...
6,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,243602,test_verzerrung_bewertung,min_score,False,False,True,Zuerst lese ich den Text im Feld „award_criter...
7,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,205270,test_verzerrung_bewertung,min_score,False,False,True,Zuerst lese ich den gesamten Text im Feld „awa...
8,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,229691,test_verzerrung_bewertung,min_score,False,False,True,1. Das Feld award_criteria enthält den Text „K...
9,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,221633,test_verzerrung_bewertung,min_score,False,False,True,Zuerst lese ich den gesamten Text im Feld „awa...


Judging 1/26
Judging 25/26
Judging 26/26

Final merged table:


,source,source_file,project_id,test_code,check_code,gt,pred,match,response,judge_backend,judge_model,judge_score,judge_raw
0,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,9583,test_verzerrung_bewertung,min_score,False,False,True,Zuerst lese ich den gesamten Text des Feldes „...,gemini_vertex,gemini-2.5-flash,1.0,1
1,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,169402,test_verzerrung_bewertung,min_score,False,False,True,"Zunächst prüfen wir, ob im Text explizit eine ...",gemini_vertex,gemini-2.5-flash,1.0,1
2,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,189879,test_verzerrung_bewertung,min_score,False,False,True,Zuerst lese ich den Text im Feld „award_criter...,gemini_vertex,gemini-2.5-flash,1.0,1
3,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,190944,test_verzerrung_bewertung,min_score,False,False,True,Zunächst lese ich den gesamten Text des Feldes...,gemini_vertex,gemini-2.5-flash,1.0,1
4,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,191846,test_verzerrung_bewertung,min_score,False,False,True,Zuerst lese ich den Text im Feld „award_criter...,gemini_vertex,gemini-2.5-flash,1.0,1
5,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,192494,test_verzerrung_bewertung,min_score,False,False,True,Die Analyse des Feldes „award_criteria“ erfolg...,gemini_vertex,gemini-2.5-flash,1.0,1
6,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,196565,test_verzerrung_bewertung,min_score,False,False,True,Die Analyse des Feldes „award_criteria“ erfolg...,gemini_vertex,gemini-2.5-flash,1.0,1
7,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,205270,test_verzerrung_bewertung,min_score,False,False,True,Zuerst lese ich den gesamten Text im Feld „awa...,gemini_vertex,gemini-2.5-flash,1.0,1
8,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,221633,test_verzerrung_bewertung,min_score,False,False,True,Zuerst lese ich den gesamten Text im Feld „awa...,gemini_vertex,gemini-2.5-flash,1.0,1
9,blackbox,uqlm_blackbox_Verzerrung_Bewertung_GPT20B.csv,229691,test_verzerrung_bewertung,min_score,False,False,True,1. Das Feld award_criteria enthält den Text „K...,gemini_vertex,gemini-2.5-flash,1.0,1


In [64]:

final_judge_table.to_csv(f"uqlm_judge_both_Verzerrung_Bewertung_GPT20B_GEMINI_2.5_flash.csv", index=False)
